In [1]:
import argparse
import os
import pickle
import time

from importlib import metadata
import torch
try:
    try:
        if metadata.version("rsl-rl"):
            raise ImportError
    except metadata.PackageNotFoundError:
        if metadata.version("rsl-rl-lib") != "3.1.1":  #2.2.4
            raise ImportError
except (metadata.PackageNotFoundError, ImportError) as e:
    raise ImportError("Please uninstall 'rsl_rl' and install 'rsl-rl-lib==2.2.4'.") from e
from rsl_rl.runners import OnPolicyRunner

In [2]:
from bp000_env_cnoid import BP000Env as RLEnv

In [3]:
# 任意設定項目
exp_name = 'collision-walking-rand'  # ckpt = 4000
ckpt = 200

action_scale = 1.0 # 動作のスケールを調整

In [4]:
## set robot path fix collisiton 
ROOT = os.path.abspath(os.path.join(os.getcwd(), ".."))  # /userdir
robot_path = os.path.join(ROOT, "userdir", "humanoid_research_k", "robots", "kawada_base.simple_collision.urdf")

In [5]:
log_dir = f"logs/{exp_name}"
env_cfg, obs_cfg, reward_cfg, command_cfg, train_cfg = pickle.load(open(f"logs/{exp_name}/cfgs.pkl", "rb"))
reward_cfg["reward_scales"] = {}

In [6]:
## override
# env_cfg["episode_length_s"] = 20.0
# command_cfg["lin_vel_x_range"] = [0.5, 0.5]
# env_cfg['dt'] = 0.01
# env_cfg['substeps'] = 10
env_cfg['base_roll_noise'] = [0,0]
env_cfg['base_pitch_noise'] = [0,0]
env_cfg['termination_if_roll_greater_than'] = 100
env_cfg['termination_if_pitch_greater_than'] = 100

In [7]:
env_cfg

{'num_actions': 12,
 'default_joint_angles': {'R_HIP_Y': 0.0,
  'R_HIP_R': 0.0,
  'R_HIP_P': -0.8,
  'R_KNEE': 1.6,
  'R_ANKLE_P': -0.8,
  'R_ANKLE_R': 0.0,
  'L_HIP_Y': 0.0,
  'L_HIP_R': 0.0,
  'L_HIP_P': -0.8,
  'L_KNEE': 1.6,
  'L_ANKLE_P': -0.8,
  'L_ANKLE_R': 0.0},
 'joint_names': ['R_HIP_Y',
  'R_HIP_R',
  'R_HIP_P',
  'R_KNEE',
  'R_ANKLE_P',
  'R_ANKLE_R',
  'L_HIP_Y',
  'L_HIP_R',
  'L_HIP_P',
  'L_KNEE',
  'L_ANKLE_P',
  'L_ANKLE_R'],
 'kp': 2000.0,
 'kd': 500.0,
 'termination_if_roll_greater_than': 100,
 'termination_if_pitch_greater_than': 100,
 'base_init_pos': [0.0, 0.0, 0.64],
 'base_init_quat': [1.0, 0.0, 0.0, 0.0],
 'episode_length_s': 20.0,
 'resampling_time_s': 4.0,
 'action_scale': 0.25,
 'simulate_action_latency': True,
 'clip_actions': 100.0,
 'dt': 0.01,
 'substeps': 10,
 'rotorInertia': 0.1,
 'base_roll_noise': [0, 0],
 'base_pitch_noise': [0, 0]}

In [8]:
env = RLEnv(
    num_envs=1,
    env_cfg=env_cfg,
    obs_cfg=obs_cfg,
    reward_cfg=reward_cfg,
    command_cfg=command_cfg,
    dt=env_cfg['dt'],
    substeps=env_cfg['substeps'],
    show_viewer=True,
    robot_urdf_path=robot_path,
)

In [9]:
runner = OnPolicyRunner(env, train_cfg, log_dir, device='cuda')
resume_path = os.path.join(log_dir, f"model_{ckpt}.pt")
runner.load(resume_path)
policy = runner.get_inference_policy(device='cuda')

obs, _ = env.reset()
cnt = 0

print("env_reset:", obs["policy"])

--------------------------------------------------------------------------------
Resolved observation sets: 
	 policy :  ['policy']
	 critic :  ['policy']
--------------------------------------------------------------------------------
Actor MLP: MLP(
  (0): Linear(in_features=45, out_features=512, bias=True)
  (1): ELU(alpha=1.0)
  (2): Linear(in_features=512, out_features=256, bias=True)
  (3): ELU(alpha=1.0)
  (4): Linear(in_features=256, out_features=128, bias=True)
  (5): ELU(alpha=1.0)
  (6): Linear(in_features=128, out_features=12, bias=True)
)
Critic MLP: MLP(
  (0): Linear(in_features=45, out_features=512, bias=True)
  (1): ELU(alpha=1.0)
  (2): Linear(in_features=512, out_features=256, bias=True)
  (3): ELU(alpha=1.0)
  (4): Linear(in_features=256, out_features=128, bias=True)
  (5): ELU(alpha=1.0)
  (6): Linear(in_features=128, out_features=1, bias=True)
)
env_reset: tensor([[0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
    

In [10]:
with torch.no_grad():
    
    actions = policy(obs)

    # アクションに倍率を適用して動きを制限
    scaled_actions = actions * action_scale

    print("Original actions : ", actions)
    print("Scaled actions : ", scaled_actions)

    obs, rews, dones, infos = env.step(scaled_actions) # スケール済みアクションを使用
    
    print(obs["policy"])
    cnt += 1


Original actions :  tensor([[-0.1029,  0.0038,  0.8968, -0.2168, -0.4307, -0.8971,  0.3350,  0.0641,
          0.6694, -0.0840, -1.4133,  0.1074]], device='cuda:0')
Scaled actions :  tensor([[-0.1029,  0.0038,  0.8968, -0.2168, -0.4307, -0.8971,  0.3350,  0.0641,
          0.6694, -0.0840, -1.4133,  0.1074]], device='cuda:0')


/userdir/irsl_rl/rl_env_base.py:110: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  self.exact_actions = torch.tensor(actions, device=self.device, dtype=torch.float32) ## copy
/userdir/irsl_rl/rl_env_cnoid.py:103: UserWarning: Creating a tensor from a list of numpy.ndarrays is extremely slow. Please consider converting the list to a single numpy.ndarray with numpy.array() before converting to a tensor. (Triggered internally at /pytorch/torch/csrc/utils/tensor_new.cpp:254.)
  self.dof_pos = torch.tensor([self.convAnglesToGenesis(sbody.angleVector())]).to(torch.float32).to(self.device)


tensor([[-3.2769e-06, -8.3160e-03, -2.0919e-06,  1.6137e-08, -1.8986e-19,
         -1.0000e+00,  1.0000e+00,  0.0000e+00,  0.0000e+00,  2.2281e-08,
          2.5473e-08, -1.3971e-04,  3.6526e-04, -1.8525e-04, -4.9388e-08,
         -9.6075e-08, -3.6861e-08, -1.3971e-04,  3.6538e-04, -1.8525e-04,
          6.3601e-07,  1.1140e-06,  1.2737e-06, -6.9785e-03,  1.8271e-02,
         -9.6273e-03, -2.4694e-06, -4.8038e-06, -1.8431e-06, -6.9813e-03,
          1.8275e-02, -9.6286e-03,  3.1800e-05, -1.0286e-01,  3.8334e-03,
          8.9682e-01, -2.1678e-01, -4.3067e-01, -8.9713e-01,  3.3501e-01,
          6.4110e-02,  6.6945e-01, -8.4019e-02, -1.4133e+00,  1.0743e-01]],
       device='cuda:0')


In [11]:
with torch.no_grad(): 
    actions = policy(obs)

    # アクションに倍率を適用して動きを制限
    scaled_actions = actions * action_scale

    print("Original actions : ", actions)
    print("Scaled actions : ", scaled_actions)

    obs, rews, dones, infos = env.step(scaled_actions) # スケール済みアクションを使用
    
    print(obs["policy"])
    print("step :", cnt)   
    cnt += 1


Original actions :  tensor([[-0.0057, -0.1900,  1.2292, -0.2825, -1.2845, -1.2822,  0.7634,  0.2681,
          0.9650, -0.3557, -2.6658,  0.0059]], device='cuda:0')
Scaled actions :  tensor([[-0.0057, -0.1900,  1.2292, -0.2825, -1.2845, -1.2822,  0.7634,  0.2681,
          0.9650, -0.3557, -2.6658,  0.0059]], device='cuda:0')
tensor([[ 2.6370e-03, -1.6469e-01, -6.5140e-02, -5.2732e-03, -2.0749e-04,
         -9.9999e-01,  1.0000e+00,  0.0000e+00,  0.0000e+00, -6.0538e-03,
          4.9583e-04,  8.7996e-03,  2.3624e-03, -1.9771e-02, -5.7505e-03,
          3.7759e-03,  3.7649e-04,  7.6706e-03,  2.8189e-03, -1.7416e-02,
         -1.1567e-03, -7.9091e-02,  6.3582e-03,  6.5760e-02,  4.5266e-02,
         -2.7328e-01, -1.4858e-01,  8.9770e-02,  2.1041e-03,  4.5583e-02,
          1.3725e-03, -6.0498e-03, -2.8009e-02, -5.7370e-03, -1.8997e-01,
          1.2292e+00, -2.8253e-01, -1.2845e+00, -1.2822e+00,  7.6343e-01,
          2.6810e-01,  9.6501e-01, -3.5572e-01, -2.6658e+00,  5.8887e-03]],
    

In [12]:
with torch.no_grad(): 
    actions = policy(obs)

    # アクションに倍率を適用して動きを制限
    scaled_actions = actions * action_scale

    print("Original actions : ", actions)
    print("Scaled actions : ", scaled_actions)

    obs, rews, dones, infos = env.step(scaled_actions) # スケール済みアクションを使用
    
    print(obs["policy"])
    print("step :", cnt)   
    cnt += 1


Original actions :  tensor([[-0.0321, -0.1006,  1.7900, -0.2380, -1.2090, -1.2044,  0.5790,  0.3906,
          0.6768, -0.4527, -2.5780,  0.2154]], device='cuda:0')
Scaled actions :  tensor([[-0.0321, -0.1006,  1.7900, -0.2380, -1.2090, -1.2044,  0.5790,  0.3906,
          0.6768, -0.4527, -2.5780,  0.2154]], device='cuda:0')
tensor([[ 4.5583e-02, -1.2005e-01, -1.5372e-01, -1.0094e-02, -3.3113e-04,
         -9.9995e-01,  1.0000e+00,  0.0000e+00,  0.0000e+00, -7.2428e-03,
         -3.2684e-04,  2.2600e-02,  1.6450e-03, -4.1647e-02, -1.6028e-02,
          1.4827e-02,  1.2296e-03,  1.7662e-02,  3.8071e-03, -5.2059e-02,
          5.3860e-03,  7.4261e-02, -2.8641e-02,  8.0508e-02, -4.4672e-02,
          8.0638e-02,  1.1273e-01,  6.8436e-02, -1.5380e-02,  5.2891e-02,
          1.5089e-02, -2.1721e-01,  8.3584e-02, -3.2077e-02, -1.0062e-01,
          1.7900e+00, -2.3801e-01, -1.2090e+00, -1.2044e+00,  5.7896e-01,
          3.9061e-01,  6.7682e-01, -4.5266e-01, -2.5780e+00,  2.1542e-01]],
    

In [13]:
with torch.no_grad(): 
    actions = policy(obs)

    # アクションに倍率を適用して動きを制限
    scaled_actions = actions * action_scale

    print("Original actions : ", actions)
    print("Scaled actions : ", scaled_actions)

    obs, rews, dones, infos = env.step(scaled_actions) # スケール済みアクションを使用
    
    print(obs["policy"])
    print("step :", cnt)   
    cnt += 1


Original actions :  tensor([[-0.0150, -0.1049,  1.4959, -0.3135, -1.3248, -1.1399,  0.5169,  0.4585,
          0.5672,  0.1700, -1.7020,  0.0835]], device='cuda:0')
Scaled actions :  tensor([[-0.0150, -0.1049,  1.4959, -0.3135, -1.3248, -1.1399,  0.5169,  0.4585,
          0.5672,  0.1700, -1.7020,  0.0835]], device='cuda:0')
tensor([[ 1.4127e-02, -9.3178e-02, -1.0212e-01, -1.4794e-02, -3.4071e-04,
         -9.9989e-01,  1.0000e+00,  0.0000e+00,  0.0000e+00, -7.2581e-03,
         -8.7963e-04,  3.8029e-02, -2.0090e-03, -4.9827e-02, -1.9626e-02,
          2.1498e-02,  3.4575e-03,  2.8157e-02,  2.5530e-03, -7.6266e-02,
          8.5035e-03, -7.8345e-02, -1.2795e-02,  8.4748e-02, -3.5349e-02,
          8.4308e-02,  7.1014e-02, -4.2759e-02, -2.1130e-04,  3.4841e-02,
         -1.0899e-02,  5.4244e-02,  1.0333e-01, -1.4965e-02, -1.0486e-01,
          1.4959e+00, -3.1348e-01, -1.3248e+00, -1.1399e+00,  5.1691e-01,
          4.5848e-01,  5.6715e-01,  1.7000e-01, -1.7020e+00,  8.3517e-02]],
    

In [14]:
with torch.no_grad(): 
    actions = policy(obs)

    # アクションに倍率を適用して動きを制限
    scaled_actions = actions * action_scale

    print("Original actions : ", actions)
    print("Scaled actions : ", scaled_actions)

    obs, rews, dones, infos = env.step(scaled_actions) # スケール済みアクションを使用
    
    print(obs["policy"])
    print("step :", cnt)   
    cnt += 1


Original actions :  tensor([[-0.2414,  0.0738,  1.6641,  0.0950, -1.1843, -0.8986,  0.0951,  0.5236,
         -0.3323,  0.0608, -1.1821,  0.0654]], device='cuda:0')
Scaled actions :  tensor([[-0.2414,  0.0738,  1.6641,  0.0950, -1.1843, -0.8986,  0.0951,  0.5236,
         -0.3323,  0.0608, -1.1821,  0.0654]], device='cuda:0')
tensor([[-4.0075e-02, -6.6415e-02, -2.0151e-01, -1.8942e-02,  7.9319e-04,
         -9.9982e-01,  1.0000e+00,  0.0000e+00,  0.0000e+00, -7.5537e-03,
         -1.2277e-03,  5.2004e-02, -3.7507e-03, -5.7176e-02, -1.9615e-02,
          2.2490e-02,  7.6367e-03,  3.3826e-02,  4.9596e-03, -9.1833e-02,
          5.2511e-03,  6.8279e-02,  2.9196e-04,  5.7038e-02,  7.0431e-03,
          1.1064e-01,  1.4601e-01, -4.5536e-02,  2.7534e-02,  2.6652e-02,
         -6.4945e-03, -1.1739e-01, -3.7979e-02, -2.4143e-01,  7.3776e-02,
          1.6641e+00,  9.5028e-02, -1.1843e+00, -8.9861e-01,  9.5053e-02,
          5.2365e-01, -3.3235e-01,  6.0823e-02, -1.1821e+00,  6.5394e-02]],
    

In [15]:
num_steps = 100
for i in range(num_steps):
    with torch.no_grad():
        if i % 100 == 0:
            print("cnt :", cnt)   
        actions = policy(obs)

        # アクションに倍率を適用して動きを制限
        scaled_actions = actions * action_scale
        
        if i % 100 == 0:
            print("Original actions : ", actions)
            print("Scaled actions : ", scaled_actions)

        obs, rews, dones, infos = env.step(scaled_actions) # スケール済みアクションを使用

        time.sleep(0.1)

        cnt += 1

cnt : 5
Original actions :  tensor([[-0.2587,  0.1837,  1.5480, -0.0726, -0.9933, -0.6832,  0.2507,  0.6792,
         -0.3859,  0.0036, -0.5479,  0.1092]], device='cuda:0')
Scaled actions :  tensor([[-0.2587,  0.1837,  1.5480, -0.0726, -0.9933, -0.6832,  0.2507,  0.6792,
         -0.3859,  0.0036, -0.5479,  0.1092]], device='cuda:0')


/userdir/irsl_rl/rl_env_base.py:110: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  self.exact_actions = torch.tensor(actions, device=self.device, dtype=torch.float32) ## copy


cnt : 105
Original actions :  tensor([[-1.7894,  1.9351,  0.9959,  0.1845, -0.1944,  0.4713,  0.6911,  1.6072,
          3.9915, -2.8065, -2.1443,  0.4806]], device='cuda:0')
Scaled actions :  tensor([[-1.7894,  1.9351,  0.9959,  0.1845, -0.1944,  0.4713,  0.6911,  1.6072,
          3.9915, -2.8065, -2.1443,  0.4806]], device='cuda:0')
cnt : 205
Original actions :  tensor([[-0.4396,  1.0377,  0.2863, -1.7246,  0.1031, -0.6841,  0.5122,  1.3877,
          5.1077, -2.0593, -1.3968,  0.7093]], device='cuda:0')
Scaled actions :  tensor([[-0.4396,  1.0377,  0.2863, -1.7246,  0.1031, -0.6841,  0.5122,  1.3877,
          5.1077, -2.0593, -1.3968,  0.7093]], device='cuda:0')
cnt : 305
Original actions :  tensor([[ 1.2447,  0.1472, -0.7641, -1.7719,  1.2632, -0.0597, -0.9161,  0.4730,
          4.5455, -0.8914,  1.1667,  0.4140]], device='cuda:0')
Scaled actions :  tensor([[ 1.2447,  0.1472, -0.7641, -1.7719,  1.2632, -0.0597, -0.9161,  0.4730,
          4.5455, -0.8914,  1.1667,  0.4140]], dev

In [16]:
env.sim.stop()